In [ ]:
# GPT-2 / WikiText-2 QAT+VQ pipeline on Kaggle GPU
# Clones the repo, runs baseline -> PTQ/QAT/QAT+VQ -> codebook fine-tune -> figures,
# then pushes results back to the gpt2-wikitext2-qatvq branch.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone -q https://github.com/abdurrahmanrussel/QAT-VQ-Compression.git repo
%cd repo
!git checkout -q gpt2-wikitext2-qatvq
!git log --oneline -3

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" scikit-learn matplotlib

In [ ]:
%cd /kaggle/working/repo/gpt2
import os
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# Step 1: baseline fine-tune. T4/P100 has 16GB -> bigger batch than the 4GB laptop run.
!python train_baseline.py --epochs 1 --bs 8

In [ ]:
# Step 2: PTQ / QAT / QAT+VQ (best-of-5-seeds, no finetune)
!python run_experiments.py --sub_dim 2 --K 256 --qat_epochs 1 --ft_lr 1e-5 --seeds 0 1 2 3 4

In [ ]:
# Step 3: codebook fine-tune on the best seed (seed 1 was best on the local run: ppl 26.44)
# Check run_experiments output above for the actual best seed on THIS run before trusting this default.
!python finetune_vq.py --seed 1 --epochs 2 --lr 5e-6

In [ ]:
!python make_figures.py
!cat artifacts/results_table.md

In [ ]:
# Push results back to GitHub. Requires a Kaggle Secret named GITHUB_TOKEN
# (fine-grained PAT, repo=QAT-VQ-Compression, Contents: Read and write).
# Add it via this kernel's editor -> Add-ons -> Secrets, before running this cell.
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

import subprocess
def sh(cmd):
    print('$', cmd.replace(token, '***') if token in cmd else cmd)
    subprocess.run(cmd, shell=True, check=True)

sh('git config user.email "abdurrahmanrussel77@gmail.com"')
sh('git config user.name "Md Abdur Rahman"')
sh('git add artifacts/results.json artifacts/results_table.md artifacts/figures/ artifacts/*.log')
sh('git commit -m "GPT-2 QAT+VQ results from Kaggle GPU" || echo "nothing to commit"')
sh(f'git push https://{token}@github.com/abdurrahmanrussel/QAT-VQ-Compression.git gpt2-wikitext2-qatvq')